# Case Study 1: Equipment Energy Consumption Anomaly Detection

Students use a power-consumption time series to connect visualization, rolling statistics, threshold-based anomaly detection, and engineering interpretation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams['figure.figsize'] = (10, 4)


## Task 1: Load the data
The included file contains hourly equipment power values. The column `anomaly_label` is included for checking your reasoning, but pretend it is unavailable when you first explore the data.

In [ ]:
df = pd.read_csv('../data/case_study_1_equipment_energy_power.csv', parse_dates=['timestamp'])
df.head()

## Task 2: Visualize the raw signal
Plot power consumption over time. Identify at least two periods that look unusual before writing any detection code.

In [ ]:
plt.plot(df['timestamp'], df['power_kW'])
plt.xlabel('Time')
plt.ylabel('Power (kW)')
plt.title('Equipment Power Consumption')
plt.show()

## Task 3: Compute rolling statistics
A rolling mean describes the local expected behavior. A rolling standard deviation describes local variability.

In [ ]:
window = 24
df['rolling_mean'] = df['power_kW'].rolling(window=window, min_periods=window).mean()
df['rolling_std'] = df['power_kW'].rolling(window=window, min_periods=window).std()
df[['timestamp','power_kW','rolling_mean','rolling_std']].head(30)

## Task 4: Detect anomalies using a z-score threshold
Change the threshold and observe how the number of detected anomalies changes.

In [ ]:
threshold = 3.0
df['z_score'] = (df['power_kW'] - df['rolling_mean']) / df['rolling_std']
df['detected_anomaly'] = df['z_score'].abs() > threshold
print('Detected anomalies:', int(df['detected_anomaly'].sum()))

In [ ]:
plt.plot(df['timestamp'], df['power_kW'], label='Power')
anom = df[df['detected_anomaly']]
plt.scatter(anom['timestamp'], anom['power_kW'], marker='x', s=60, label='Detected anomaly')
plt.xlabel('Time')
plt.ylabel('Power (kW)')
plt.legend()
plt.title('Threshold-Based Anomaly Detection')
plt.show()

## Task 5: Evaluate and interpret
Use the provided label only after you complete the visual and threshold analysis. In a real system, labels may be unavailable or incomplete.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
valid = df['detected_anomaly'].notna()
print(confusion_matrix(df.loc[valid,'anomaly_label'], df.loc[valid,'detected_anomaly'].astype(int)))
print(classification_report(df.loc[valid,'anomaly_label'], df.loc[valid,'detected_anomaly'].astype(int), zero_division=0))

## Reflection
1. What may cause a high-power anomaly in a manufacturing facility?
2. What may cause a low-power anomaly?
3. Which is more harmful: a false alarm or a missed anomaly? Explain using an engineering argument.